# Dataset 6 — Inspection
**Danish adolescents — experimental longitudinal study with baseline + intervention phases**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW6 = Path("../data_raw/6. data set, Danish adolescents/Data")
SESSION  = RAW6 / "behavioral data/session_time_dataset_97_5_cutoff_data_forwarded.csv"
BASELINE = RAW6 / "behavioral data/baseline_sessions_filt3.csv"
DEMO     = RAW6 / "behavioral data/demographic_data.csv"
SURVEY_C = RAW6 / "behavioral data/survey_data_clean.csv"
SURVEY_F = RAW6 / "behavioral data/survey_data_full.csv"
PRESURVEY = RAW6 / "Danish adolescents & one sec, main study, pre-survey.csv"
print("Paths configured")

## 1. File Inventory

In [ ]:
import os
root_dir = Path("../data_raw/6. data set, Danish adolescents")
for root, dirs, files in os.walk(root_dir):
    level = Path(root).relative_to(root_dir)
    for f in files:
        fpath = Path(root) / f
        size_mb = fpath.stat().st_size / 1e6
        print(f"  {level}/{f}  ({size_mb:.1f} MB)")

## 2. Session File — Structure (Primary Behavioral)

In [ ]:
sess = pd.read_csv(SESSION, low_memory=False)
print("Shape:", sess.shape)
print("\nColumns:", sess.columns.tolist())
sess.head(3)

## 3. Delimiter Confirmation

In [ ]:
with open(SESSION) as f:
    header = f.readline()
print("Commas in header:", header.count(','))
print("Semicolons in header:", header.count(';'))
print("Header sample:", header[:150])

## 4. Timestamp Format — Session File

In [ ]:
print("Timestamp samples:", sess['timestamp'].head(5).tolist())
parsed = pd.to_datetime(sess['timestamp'], utc=True, errors='coerce')
print("Parse failures:", parsed.isna().sum())
print("Sample parsed:", parsed.head(3).tolist())

## 5. Resolution — Session File

In [ ]:
print("resolution unique:", sess['resolution'].value_counts(dropna=False).to_dict())
print("\nNote: ALL events are openedApp in this file.")

## 6. Period and Condition

In [ ]:
print("period:", sess['period'].value_counts(dropna=False).to_dict())
print("\ncondition (text):", sess['condition'].value_counts(dropna=False).to_dict())
print("\ncond (numeric):", sess['cond'].value_counts(dropna=False).sort_index().to_dict())

## 7. Intervention Columns

In [ ]:
print("interventionType:", sess['interventionType'].value_counts(dropna=False).head(8).to_dict())
print()
print("interventionDuration top values:", sess['interventionDuration'].value_counts(dropna=False).head(5).to_dict())
print()
print("isReIntervention:", sess['isReIntervention'].value_counts(dropna=False).to_dict())
print()
print("timeIntervalUntilReIntervention top:", sess['timeIntervalUntilReIntervention'].value_counts(dropna=False).head(5).to_dict())

## 8. Participant IDs — Session File

In [ ]:
print("N unique IDs:", sess['ID'].nunique())
print("Sample IDs:", sess['ID'].head(5).tolist())
print("ID length range:", sess['ID'].dropna().str.len().unique().tolist())

## 9. Baseline File — Structure

In [ ]:
base = pd.read_csv(BASELINE, low_memory=False)
print("Shape:", base.shape)
print("period:", base['period'].value_counts(dropna=False).to_dict())
print("resolution:", base['resolution'].value_counts(dropna=False).to_dict())
print("studyActivated:", base['studyActivated'].value_counts(dropna=False).to_dict())
print("N unique IDs:", base['ID'].nunique())
print("\nExtra columns in baseline vs session:")
print([c for c in base.columns if c not in sess.columns])

## 10. ID Overlap Across Files

In [ ]:
demo    = pd.read_csv(DEMO, low_memory=False)
surv_c  = pd.read_csv(SURVEY_C, low_memory=False)

sess_ids = set(sess['ID'].unique())
base_ids = set(base['ID'].unique())
demo_ids = set(demo['ID'].unique())
surv_ids = set(surv_c['response_id'].unique())

print(f"Session IDs: {len(sess_ids)}")
print(f"Baseline IDs: {len(base_ids)}")
print(f"Demo IDs: {len(demo_ids)}")
print(f"Survey clean IDs: {len(surv_ids)}")
print(f"\nSession ∩ Demo: {len(sess_ids & demo_ids)}")
print(f"Session ∩ Survey: {len(sess_ids & surv_ids)}")
print(f"Demo ∩ Survey: {len(demo_ids & surv_ids)}")
print(f"Session ∩ Demo ∩ Survey: {len(sess_ids & demo_ids & surv_ids)}")
print(f"\nDemo only (no behavioral): {len(demo_ids - sess_ids)}")

## 11. Demographics

In [ ]:
print("Demo shape:", demo.shape)
print("\nage:", demo['age'].value_counts(dropna=False).sort_index().to_dict())
print("\ngender:", demo['gender'].value_counts(dropna=False).to_dict())
print("\nregion:", demo['region'].value_counts(dropna=False).to_dict())

## 12. Survey Clean — Structure

In [ ]:
print("Shape:", surv_c.shape)
print("Columns:", surv_c.columns.tolist())
print("\nsurvey_nr:", surv_c['survey_nr'].value_counts(dropna=False).sort_index().to_dict())
print("\ncond:", surv_c['cond'].value_counts(dropna=False).to_dict())
print("\ncond_exp2:", surv_c['cond_exp2'].value_counts(dropna=False).to_dict())
print("\nUnique response_id:", surv_c['response_id'].nunique())

## 13. Survey Full — Comparison

In [ ]:
surv_f = pd.read_csv(SURVEY_F, low_memory=False)
print("survey_full shape:", surv_f.shape)
print("survey_nr:", surv_f['survey_nr'].value_counts(dropna=False).sort_index().to_dict())
print("Unique response_id:", surv_f['response_id'].nunique())
print("\nExtra cols in full vs clean:")
print([c for c in surv_f.columns if c not in surv_c.columns])

## 14. Sequence Columns — Session File

In [ ]:
seq_cols = ['app_next','app_next2','app_next3','app_prev','app_prev2',
            'resolution_next','resolution_next2','resolution_next3',
            'group_id','first_in_group','last_close_in_group',
            'time_diff_secs','time_diff_minutes']
for col in seq_cols:
    if col in sess.columns:
        n_null = sess[col].isna().sum()
        print(f"{col}: {n_null} NaN ({n_null/len(sess)*100:.1f}%)")

## 15. Full Null Summary — Session File

In [ ]:
null_pct = (sess.isnull().sum() / len(sess) * 100).round(1)
pd.DataFrame({'null_count': sess.isnull().sum(), 'null_pct': null_pct}).sort_values('null_count', ascending=False).head(20)

## 16. Duplicate Rows Check

In [ ]:
for name, df in [('session', sess), ('baseline', base), ('demo', demo), ('survey_clean', surv_c)]:
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

## 17. group_id — Session Structure

In [ ]:
print("group_id non-null:", sess['group_id'].notna().sum())
print("Unique group_ids:", sess['group_id'].nunique())
print("Events per group_id (top 5):")
print(sess['group_id'].value_counts().head(5))

## 18. Participation Day and Period

In [ ]:
print("participation_day range:", sess['participation_day'].min(), "–", sess['participation_day'].max())
print("\nperiod × condition cross-tab (counts):")
print(pd.crosstab(sess['period'], sess['condition']))

## 19. Inspection Summary

In [ ]:
print("""
=== DS6 INSPECTION SUMMARY ===

Primary behavioral file: session_time_dataset_97_5_cutoff_data_forwarded.csv
  353,796 rows × 70 cols — comma delimiter
  ALL rows: resolution = openedApp (no dismiss/close)
  Periods: intervention (194,907), baseline (158,889)
  Conditions: control (153,236), default (100,888), intervention (99,672)
  243 unique participants (ID = 15-char alphanumeric)
  Timestamp: ISO 8601 UTC — parse directly
  interventionType: denmark (68K), breathingExercise (39K), math (2.7K), followTheDot (1.5K)
  isReIntervention: 2,120 events

Baseline file: baseline_sessions_filt3.csv
  195,515 rows × 63 cols — baseline period only, openedApp only
  Contains date2 and close_ts for session matching

Demographics: 399 rows × 4 cols
  Ages 13–17 (true adolescent range)
  Gender: kvinde/mand (female/male)
  All 243 behavioral participants are in demographics

Survey clean: 478 rows, 269 unique participants, 2 waves
Survey full: 929 rows, 440 unique participants, 3 waves (includes raw items)
Pre-survey (Qualtrics): 467 rows, 68+ cols — not yet cleaned

ID alignment:
  Session IDs (243) ⊂ Survey IDs (269) ⊂ Demo IDs (399) — 130 demographic participants have neither behavioral nor survey data
  Direct join on ID = response_id (no transformation needed)
  156 participants in demo have no behavioral data

Open items:
  - Pre-survey Qualtrics file not yet cleaned
  - survey_data_full.csv wave 3 not represented in clean file
  - Session close times inferred from group_id + time_diff_secs
  - purpose: 89% NaN
""")